# The Role of Data Science in Optimizing Renewable Energy Generation from Wind Farms

Prototype notebook for Research Methodology Lab Experiment 7.

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_excel('../data/cleaned_data.xlsx')
df['time_stamp'] = pd.to_datetime(df['time_stamp'])
df = df.sort_values('time_stamp').reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
FEATURES = ['wind_speed_3_avg','wind_speed_4_avg','sensor_15_avg','sensor_16_avg','sensor_23_avg','sensor_24_avg','sensor_25_avg','sensor_45','sensor_50']
TARGET = 'power_29_avg'
train_df = df[df.train_test.eq('train')].dropna(subset=FEATURES+[TARGET]).copy()
pred_df = df[df.train_test.eq('prediction')].dropna(subset=FEATURES).copy()
print('train:', len(train_df), 'prediction:', len(pred_df))

In [ ]:
split = int(len(train_df)*0.80)
tr, va = train_df.iloc[:split], train_df.iloc[split:]
models = {'Linear Regression': LinearRegression(), 'Random Forest': RandomForestRegressor(n_estimators=250,max_depth=18,min_samples_leaf=2,random_state=42,n_jobs=-1)}
rows=[]
for name,m in models.items():
    m.fit(tr[FEATURES], tr[TARGET])
    p=m.predict(va[FEATURES])
    rows.append([name, mean_absolute_error(va[TARGET],p), np.sqrt(mean_squared_error(va[TARGET],p)), r2_score(va[TARGET],p)])
metrics=pd.DataFrame(rows,columns=['model','MAE','RMSE','R2'])
metrics

In [ ]:
final_model = models['Random Forest']
final_model.fit(train_df[FEATURES], train_df[TARGET])
pred = final_model.predict(pred_df[FEATURES])
out = pred_df[['time_stamp','asset_id','id']].copy()
out['predicted_power_29_avg'] = pred
out.to_csv('../results/predictions.csv', index=False)
out.head()

## Conclusion
The prototype demonstrates how cleaned wind-farm sensor data can be transformed into a reproducible machine-learning workflow for wind-power forecasting. The predictions can support generation planning and operational decision-making.